# Geospatial Delivery Hotspot Clustering

Detect dense delivery zones and geographic noise from latitude-longitude events.

**Portfolio category:** Geospatial clustering

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Privacy-safe delivery coordinates

In [ ]:
centres = np.array([
    [17.3850, 78.4867],
    [17.4435, 78.3772],
    [17.4483, 78.3915],
    [17.4933, 78.3995],
])
rows = []
for centre_id, centre in enumerate(centres):
    points = rng.normal(centre, [0.006, 0.007], size=(180, 2))
    frame = pd.DataFrame(points, columns=["latitude", "longitude"])
    frame["hidden_centre"] = centre_id
    rows.append(frame)
noise = pd.DataFrame({
    "latitude": rng.uniform(17.30, 17.56, 70),
    "longitude": rng.uniform(78.30, 78.58, 70),
    "hidden_centre": -1,
})
deliveries = pd.concat([*rows, noise], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
display(deliveries.head())

## 3. Coordinate quality

In [ ]:
display(deliveries[["latitude", "longitude"]].describe().T)
print("Duplicate coordinates:", int(deliveries.duplicated(["latitude", "longitude"]).sum()))

## 4. Haversine DBSCAN

In [ ]:
earth_radius_km = 6371.0088
coordinates_radians = np.radians(deliveries[["latitude", "longitude"]])
epsilon_km = 1.6
model = DBSCAN(
    eps=epsilon_km / earth_radius_km,
    min_samples=14,
    metric="haversine",
)
deliveries["cluster"] = model.fit_predict(coordinates_radians)

## 5. Hotspot quality

In [ ]:
non_noise = deliveries["cluster"] >= 0
cluster_count = deliveries.loc[non_noise, "cluster"].nunique()
score = silhouette_score(
    deliveries.loc[non_noise, ["latitude", "longitude"]],
    deliveries.loc[non_noise, "cluster"],
) if cluster_count > 1 else np.nan
display(pd.Series({
    "hotspots": cluster_count,
    "noise_rate": 1 - non_noise.mean(),
    "non_noise_silhouette": score,
}).to_frame("value"))

## 6. Hotspot profiles

In [ ]:
hotspots = deliveries.query("cluster >= 0").groupby("cluster").agg(
    deliveries=("cluster", "size"),
    centre_latitude=("latitude", "mean"),
    centre_longitude=("longitude", "mean"),
).sort_values("deliveries", ascending=False)
display(hotspots.round(5))

## 7. Map-like scatter plot

In [ ]:
sns.scatterplot(
    data=deliveries,
    x="longitude",
    y="latitude",
    hue="cluster",
    palette="tab10",
    s=28,
    alpha=0.75,
)
plt.title("Delivery hotspots; cluster -1 is noise")
plt.tight_layout()

## 8. Operational prioritisation

In [ ]:
hotspots["share_of_all_deliveries"] = hotspots["deliveries"] / len(deliveries)
display(hotspots.head(10).round(4))

## 9. Key findings

Epsilon is a business and geographic choice: calibrate it in kilometres and test stability across time windows.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For geospatial delivery hotspot clustering,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.